# Collect area forecasts
Run top to bottom once, then schedule hourly. Requires `workspace.default.live_station_info`
from the station poller. This creates two reference tables and appends forecast history in your own workspace.
GitHub shares this code, not previously collected data.

The 0.02-degree grid is our grouping of request points, not the weather model's grid.
Forecast snapshots record retrieval time; it is not a model-run initialization or publication time.
Rerunning only the append cell duplicates that downloaded snapshot.

In [ ]:
import time
import pandas as pd
import requests

spark.conf.set("spark.sql.session.timeZone", "UTC")
forecast_table = "workspace.default.weather_forecast_history"
grid_size = 0.02

url = "https://api.open-meteo.com/v1/forecast"
weather_variables = [
    "temperature_2m", "apparent_temperature", "relative_humidity_2m",
    "precipitation", "snowfall", "wind_speed_10m", "wind_gusts_10m",
    "cloud_cover", "visibility"
]
params = {
    "hourly": ",".join(weather_variables),
    "models": "gfs_seamless",
    "forecast_hours": 24,
    "timezone": "GMT",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

In [ ]:
# Latest coordinates for each station.
station_locations = spark.sql("""
    SELECT station_id, name, lat, lon
    FROM (
        SELECT
            station_id,
            name,
            lat,
            lon,
            ROW_NUMBER() OVER (
                PARTITION BY station_id
                ORDER BY fetched_at DESC
            ) AS row_number
        FROM workspace.default.live_station_info
    )
    WHERE row_number = 1
      AND lat IS NOT NULL
      AND lon IS NOT NULL
""").toPandas()

print("Stations:", len(station_locations))

if station_locations.empty:
    raise ValueError("No station coordinates available. Run the station-information poller first.")

In [ ]:
# establish a grid for lat and long to request from open meteo



station_locations["forecast_lat"] = (
    (station_locations["lat"] / grid_size).round() * grid_size
).round(2)

station_locations["forecast_lon"] = (
    (station_locations["lon"] / grid_size).round() * grid_size
).round(2)

station_locations["forecast_location_id"] = (
    station_locations["forecast_lat"].astype(str)
    + "_"
    + station_locations["forecast_lon"].astype(str)
)

# use the list of stations to create a list of requested forecast areas to request from open-meteo

forecast_locations = (
    station_locations[
        ["forecast_location_id", "forecast_lat", "forecast_lon"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Forecast request locations:", len(forecast_locations))

In [ ]:
# save the locations and station mapping

station_mapping = station_locations[
    ["station_id", "forecast_location_id"]
].copy()

spark.createDataFrame(forecast_locations).write.mode("overwrite").saveAsTable(
    "workspace.default.weather_forecast_locations"
)

spark.createDataFrame(station_mapping).write.mode("overwrite").saveAsTable(
    "workspace.default.station_forecast_mapping"
)

print("Forecast locations saved:", len(forecast_locations))
print("Station mappings saved:", len(station_mapping))

In [ ]:
# Download each location and keep each retrieval timestamp.


forecast_frames = []

for _, location in forecast_locations.iterrows():
    location_params = params.copy()
    location_params["latitude"] = location["forecast_lat"]
    location_params["longitude"] = location["forecast_lon"]

    response = requests.get(
        url, params=location_params, timeout=60
    )
    response.raise_for_status()

    retrieved_at = pd.Timestamp.now(tz="UTC")
    data = response.json()

    forecast = pd.DataFrame(data["hourly"])

    forecast = forecast.rename(columns={"time": "valid_at_utc"})
    forecast["valid_at_utc"] = pd.to_datetime(
        forecast["valid_at_utc"], utc=True
    )

    forecast["forecast_location_id"] = location["forecast_location_id"]
    forecast["retrieved_at_utc"] = retrieved_at
    forecast["model"] = location_params["models"]

    forecast["grid_latitude"] = data["latitude"]
    forecast["grid_longitude"] = data["longitude"]

    forecast_frames.append(forecast)

    time.sleep(1)

all_forecasts = pd.concat(forecast_frames, ignore_index=True)
print("Weather units:", data["hourly_units"])

print("Locations collected:",
      all_forecasts["forecast_location_id"].nunique())
print("Forecast rows:", len(all_forecasts))

In [ ]:
# save the forecasts to a Delta table





(
    spark.createDataFrame(all_forecasts)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(forecast_table)
)

print("Saved forecast rows:", len(all_forecasts))